<a href="https://colab.research.google.com/github/nico89h/Analisis-computacional-de-datos-de-microbioma-intestinal-mediante-grafos-y-aprendizaje-automatico/blob/main/Agrupacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Análisis computacional de datos de microbioma intestinal mediante grafos y aprendizaje automático**


En este bloque se identificara a cada secuencia de ADN md5 con su respectiva taxonomia.

Se usaran dos archivos matriz_final.tsv ( Es la matriz de abundancia cruda de cada paciente, es decir, con los hash)

Luego, tenemos taxonomy.tsv, que corresponde al diccionario de cada hash md5# Nueva sección

In [1]:
import pandas as pd
import numpy as np

matriz= pd.read_csv('/content/drive/MyDrive/Tesis/desarrollo/matriz_final.tsv', sep = '\t', skiprows=1)
matriz= matriz.rename(columns={'#OTU ID': 'Feature ID'})
taxonomia = pd.read_csv('/content/drive/MyDrive/Tesis/desarrollo/taxonomy.tsv', sep='\t')
#tenemos que en matriz, cada una de las columnas son los pacientes, y las filas son los hash
#en taxonomia, tendremos: fila(hash) | taxonomia | confianza( de que sea esa bacteria)
# tenemos que los hash NO SE REPITEN, son UNICOS, no hay filas repetidas

#comienzo eliminando lo innecesario de la 3era columna ya que solo me importa el genero de la bacteria, no toda
#la informacion adicional

def extraer_nombre_bacteria(tax_string):
    # Si la celda está vacía o sin asignar
    if pd.isna(tax_string) or tax_string == 'Unassigned':
        return 'Desconocida'

    # Cortamos el string en una lista de niveles
    niveles = tax_string.split(';')

    # 1. EL FILTRO: Eliminamos por completo la especie ('s__') si es que existe
    niveles_sin_especie = [nivel for nivel in niveles if not nivel.strip().startswith('s__')]

    # 2. BÚSQUEDA: Recorremos lo que quedó de atrás para adelante
    for nivel in reversed(niveles_sin_especie):
        nivel_limpio = nivel.strip()

        # Buscamos el separador biológico
        if '__' in nivel_limpio:
            nombre_puro = nivel_limpio.split('__')[-1].strip()

            # Si el nivel no está vacío (ej. no es solo "g__"), lo retornamos
            if nombre_puro:
                return nombre_puro

    # Fallback de seguridad
    return 'Desconocida'


taxonomia['Bacteria_Limpia'] = taxonomia['Taxon'].apply(extraer_nombre_bacteria)

#luego hago un inner join, usando los hash

df_fusionado = pd.merge(taxonomia[['Feature ID', 'Bacteria_Limpia']], matriz, on='Feature ID', how='inner')

df_fusionado = df_fusionado.drop(columns=['Feature ID'])

# Agrupamos por el nombre de la bacteria y sumamos los valores numéricos de los pacientes
df_fase1 = df_fusionado.groupby('Bacteria_Limpia').sum().reset_index()

# Verificamos el resultado final
print("Dimensiones de la matriz agrupada:", df_fase1.shape)
print(df_fase1.head())
df_fase1.to_csv('/content/drive/MyDrive/Tesis/desarrollo/matriztotal.tsv', sep='\t', index=False)


Dimensiones de la matriz agrupada: (403, 122)
  Bacteria_Limpia  SRR10442269_1  SRR10442270_1  SRR10442271_1  SRR10442272_1  \
0           11-24            0.0            0.0            0.0            0.0   
1          4-29-1            0.0            0.0            0.0            0.0   
2              A2            0.0            0.0            0.0            0.0   
3  ADurb.Bin063-1            0.0            0.0            0.0            0.0   
4        AKAU4049            0.0            0.0            0.0            0.0   

   SRR10442273_1  SRR10442274_1  SRR10442275_1  SRR10442276_1  SRR10442277_1  \
0            0.0            0.0            0.0            0.0            0.0   
1            0.0            0.0            0.0            0.0            0.0   
2            0.0            0.0            0.0            0.0            0.0   
3            0.0            0.0            0.0            0.0            0.0   
4            0.0            0.0            0.0            0.0      

**Se define la relacion de los nodos y se hace la Normalización Logarítmica (Log1p)**

In [ ]:
# Se utilizara la funcion melt, para poder definir la relacion entre los nodos y el "peso" de la arista
# Se crea la tabla pivotante
# Se van a tener 3 columnas : Bacteria | Paciente | AbundanciaTaxonomica
# En esta etapa tmb se eliminan ceros
import numpy as np

df_fase1=df_fase1.melt(id_vars=['Bacteria_Limpia'], var_name='Paciente', value_name='AbundanciaTaxonomica')
#print(df_fase1)

#elimino los valores de 0.0
df_fase1.drop(df_fase1[df_fase1['AbundanciaTaxonomica'] == 0.0].index, inplace=True)

df_fase1['AbundanciaTaxonomica']= np.log1p(df_fase1['AbundanciaTaxonomica'])
print(df_fase1)


                       Bacteria_Limpia       Paciente  AbundanciaTaxonomica
20                         Akkermansia  SRR10442269_1              1.098612
41                        Anaerostipes  SRR10442269_1              3.433987
56                         Bacteroides  SRR10442269_1              6.555357
60                     Bifidobacterium  SRR10442269_1              5.575949
63                             Blautia  SRR10442269_1              5.746203
...                                ...            ...                   ...
48754   [Eubacterium]_ventriosum_group  SRR10442395_1              3.367296
48756  [Ruminococcus]_gauvreauii_group  SRR10442395_1              5.257495
48757      [Ruminococcus]_gnavus_group  SRR10442395_1              7.505492
48758     [Ruminococcus]_torques_group  SRR10442395_1              8.117909
48762                       uncultured  SRR10442395_1              4.406719

[11261 rows x 3 columns]


**Construcción del Grafo Heterogéneo**

In [ ]:
#se comienza con un diccionario, sabiendo cuales pacientes tienen TEA y son neurotipicos

